# MNIST Digit Classification with CNN

Train a simple convolutional neural network on the MNIST dataset using PyTorch (CPU only).

## 1. Imports

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np

device = torch.device("cpu")
print(f"Using device: {device}")

ModuleNotFoundError: No module named 'torch'

## 2. Data Preprocessing & Augmentation

- **Training**: Random affine (translation, scaling, rotation) + standard normalization (mean=0.1307, std=0.3081)
- **Test**: Standard normalization only

In [ ]:
# MNIST standard normalization constants
MNIST_MEAN = 0.1307
MNIST_STD = 0.3081

train_transform = transforms.Compose([
    transforms.RandomAffine(
        degrees=15,           # rotation up to ±15°
        translate=(0.1, 0.1), # translation up to 10% in each direction
        scale=(0.85, 1.15),   # scaling between 85%-115%
    ),
    transforms.ToTensor(),
    transforms.Normalize((MNIST_MEAN,), (MNIST_STD,)),
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((MNIST_MEAN,), (MNIST_STD,)),
])

## 3. Download MNIST & Create DataLoaders

In [ ]:
BATCH_SIZE = 128

train_dataset = datasets.MNIST(root="./data", train=True, download=True, transform=train_transform)
test_dataset = datasets.MNIST(root="./data", train=False, download=True, transform=test_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Training samples: {len(train_dataset)}")
print(f"Test samples:     {len(test_dataset)}")
print(f"Image shape:      {train_dataset[0][0].shape}")

## 4. Visualize Sample Training Images

In [ ]:
fig, axes = plt.subplots(2, 8, figsize=(12, 3))
for i, ax in enumerate(axes.flat):
    img, label = train_dataset[i]
    # Undo normalization for display
    img = img.squeeze() * MNIST_STD + MNIST_MEAN
    ax.imshow(img, cmap="gray")
    ax.set_title(str(label), fontsize=10)
    ax.axis("off")
plt.suptitle("Sample Training Images (with augmentation)")
plt.tight_layout()
plt.show()

## 5. Define CNN Model

Architecture: Conv(1→32) → Conv(32→64) → MaxPool → Conv(64→128) → MaxPool → FC(128) → FC(10)

In [ ]:
from model import DigitCNN

model = DigitCNN().to(device)
print(model)
total_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal parameters: {total_params:,}")

## 6. Training & Evaluation Loop

Cross-entropy loss, Adam optimizer, 5 epochs with test accuracy after each epoch.

In [ ]:
def evaluate(model, loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
    return correct / total


criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

NUM_EPOCHS = 5
train_losses = []
test_accuracies = []

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    running_loss = 0.0
    for batch_idx, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    avg_loss = running_loss / len(train_loader)
    train_losses.append(avg_loss)

    test_acc = evaluate(model, test_loader)
    test_accuracies.append(test_acc)

    print(f"Epoch {epoch}/{NUM_EPOCHS} — Train Loss: {avg_loss:.4f} — Test Accuracy: {test_acc*100:.2f}%")

## 7. Plot Training Results

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

ax1.plot(range(1, NUM_EPOCHS + 1), train_losses, "o-")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.set_title("Training Loss")
ax1.grid(True)

ax2.plot(range(1, NUM_EPOCHS + 1), [a * 100 for a in test_accuracies], "o-", color="green")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy (%)")
ax2.set_title("Test Accuracy")
ax2.grid(True)

plt.tight_layout()
plt.show()

print(f"\nFinal test accuracy: {test_accuracies[-1]*100:.2f}%")